# HARI 1 (30 September 2026)

In [0]:
# Nama : Davin Zoe Akhmad (3326600003)
# Prodi: Sains Data Terapan A 2026

import pandas as pd

SUMBER = "/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/jumlah-wisatawan-tahun-n-2025.csv"
df = pd.read_csv(SUMBER)
df.head()

,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,kode_wisata,nama_wisata,januari,februari,maret,april,mei,juni,juli,agustus,september,oktober,november,desember,total,satuan,tahun
0,35,JAWA TIMUR,3516,KABUPATEN MOJOKERTO,1.1,Wana Wisata Padusan Pacet,36183,22336,7343,39272,34426,32053,31179,24808,27039,29456,32362,35487,351944,orang,2025
1,35,JAWA TIMUR,3516,KABUPATEN MOJOKERTO,1.2,Kolam/Pemandian Air Panas,18521,10066,2586,14203,13918,14465,14646,11187,12298,12970,14820,17234,156914,orang,2025
2,35,JAWA TIMUR,3516,KABUPATEN MOJOKERTO,1.3,Dlundung,9777,4090,1358,7890,7903,6793,8102,6490,7669,7931,8157,9409,85569,orang,2025
3,35,JAWA TIMUR,3516,KABUPATEN MOJOKERTO,1.4,Jolotundo,5673,4476,1752,4005,5945,9721,8145,6054,6212,8248,7141,6981,74353,orang,2025
4,35,JAWA TIMUR,3516,KABUPATEN MOJOKERTO,1.5,Makam Religius Troloyo,28540,37483,5234,11503,20851,17526,29813,14153,20358,21873,23597,28317,259248,orang,2025


In [0]:
df = df[df["tahun"] == 2025]

df["rata_per_bulan"] = (df["total"] / 12).round()

def beri_kategori(rata):
    if rata > 20000:
        return "Tinggi"
    elif rata >= 10000:
        return "Sedang"
    return "Rendah"

df["kategori"] = df["rata_per_bulan"].apply(beri_kategori)

hasil = df[["nama_wisata", "total", "rata_per_bulan", "kategori"]] \
          .sort_values("total", ascending=False)
display(hasil.head())

nama_wisata,total,rata_per_bulan,kategori
Wana Wisata Padusan Pacet,351944,29329.0,Tinggi
Makam Religius Troloyo,259248,21604.0,Tinggi
Kolam/Pemandian Air Panas,156914,13076.0,Sedang
Dlundung,85569,7131.0,Rendah
Jolotundo,74353,6196.0,Rendah


In [0]:
from pyspark.sql import functions as F

SUMBER = "/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/jumlah-wisatawan-tahun-n-2025.csv"

df = spark.read.csv(SUMBER, header=True, inferSchema=True)
df = df.filter(F.col("tahun") == 2025)

df = df.withColumn("rata_per_bulan", F.round(F.col("total") / 12))

# macet di sini: bagaimana membuat kolom berdasarkan kondisi?
# di pandas saya pakai if-else di dalam fungsi, di Spark saya tidak tahu padanannya.
df = df.withColumn("kategori", if F.col("rata_per_bulan") > 20000 ...)

  File <command-7107102329775016>, line 12
    df = df.withColumn("kategori", if F.col("rata_per_bulan") > 20000 ...)
                                   ^
SyntaxError: invalid syntax


---

# HARI 2 (1 Oktober 2026)

In [0]:
from pyspark.sql import functions as F

SUMBER = "/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/jumlah-wisatawan-tahun-n-2025.csv"
df = spark.read.csv(SUMBER, header=True, inferSchema=True)

df.printSchema()

root
 |-- kode_provinsi: integer (nullable = true)
 |-- nama_provinsi: string (nullable = true)
 |-- kode_kabupaten_kota: integer (nullable = true)
 |-- nama_kabupaten_kota: string (nullable = true)
 |-- kode_wisata: double (nullable = true)
 |-- nama_wisata: string (nullable = true)
 |-- januari: integer (nullable = true)
 |-- februari: integer (nullable = true)
 |-- maret: integer (nullable = true)
 |-- april: integer (nullable = true)
 |-- mei: integer (nullable = true)
 |-- juni: integer (nullable = true)
 |-- juli: integer (nullable = true)
 |-- agustus: integer (nullable = true)
 |-- september: integer (nullable = true)
 |-- oktober: integer (nullable = true)
 |-- november: integer (nullable = true)
 |-- desember: integer (nullable = true)
 |-- total: integer (nullable = true)
 |-- satuan: string (nullable = true)
 |-- tahun: integer (nullable = true)



In [0]:
display(df.select("nama_wisata", "total"))

nama_wisata,total
Wana Wisata Padusan Pacet,351944
Kolam/Pemandian Air Panas,156914
Dlundung,85569
Jolotundo,74353
Makam Religius Troloyo,259248
Museum Trowulan,62236
Ubalan,22627
Coban Canggu,13405
Candi Bajang Ratu,23842
Candi Brahu,26407


In [0]:
display(df.filter(F.col("total") > 100000).select("nama_wisata", "total"))

nama_wisata,total
Wana Wisata Padusan Pacet,351944
Kolam/Pemandian Air Panas,156914
Makam Religius Troloyo,259248


In [0]:
display(df.orderBy(F.col("total").desc()).select("nama_wisata", "total"))

nama_wisata,total
Wana Wisata Padusan Pacet,351944
Makam Religius Troloyo,259248
Kolam/Pemandian Air Panas,156914
Dlundung,85569
Jolotundo,74353
Museum Trowulan,62236
Candi Brahu,26407
Candi Bajang Ratu,23842
Ubalan,22627
Candi Tikus,19239


In [0]:
display(df.groupBy("tahun").agg(F.count("*").alias("jumlah_tempat")))

tahun,jumlah_tempat
2025,12
2024,1


In [0]:
# latihan kondisi (if-else) dengan when/otherwise
display(
    df.withColumn(
        "kategori",
        F.when(F.col("total") > 100000, "Tinggi").otherwise("Lainnya"))
        .select("nama_wisata", "total", "kategori")
)

nama_wisata,total,kategori
Wana Wisata Padusan Pacet,351944,Tinggi
Kolam/Pemandian Air Panas,156914,Tinggi
Dlundung,85569,Lainnya
Jolotundo,74353,Lainnya
Makam Religius Troloyo,259248,Tinggi
Museum Trowulan,62236,Lainnya
Ubalan,22627,Lainnya
Coban Canggu,13405,Lainnya
Candi Bajang Ratu,23842,Lainnya
Candi Brahu,26407,Lainnya


---

# HARI 3 (2 Oktober 2026)

In [0]:
from pyspark.sql import functions as F

SUMBER = "/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/jumlah-wisatawan-tahun-n-2025.csv"

df = spark.read.csv(SUMBER, header=True, inferSchema=True)
df = df.filter(F.col("tahun") == 2025)

hasil = (
    df
    .withColumn("rata_per_bulan", F.round(F.col("jumlah_total") / 12))
    .withColumn("kategori",
                F.when(F.col("rata_per_bulan") > 20000, "Tinggi")
                 .when(F.col("rata_per_bulan") >= 10000, "Sedang")
                 .otherwise("Rendah"))
    .orderBy(F.col("total").desc()))

hasil.write.mode("overwrite")\
     .parquet(f"/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/output/wisata_ringkas")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5298457253265916>, line 18
      6 df = df.filter(F.col("tahun") == 2025)
      8 hasil = (
      9     df
     10     .withColumn("rata_per_bulan", F.round(F.col("jumlah_total") / 12))
   (...)     14                  .otherwise("Rendah"))
     15     .orderBy(F.col("total").desc()))
     17 hasil.write.mode("overwrite")\
---> 18      .parquet(f"/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/output/wisata_ringkas")

File <command-5298457253265916>, line 9
      5 df = spark.read.csv(SUMBER, header=True, inferSchema=True)
      6 df = df.filter(F.col("tahun") == 2025)
      8 hasil = (
----> 9     df
     10     .withColumn("rata_per_bulan", F.round(F.col("jumlah_total") / 12))
     11     .withColumn("kategori",
     12                 F.when(F.col("rata_per_bulan") > 20000, "Tinggi")
     13                  .w

In [0]:
from pyspark.sql import functions as F

SUMBER = "/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/jumlah-wisatawan-tahun-n-2025.csv"

df = spark.read.csv(SUMBER, header=True, inferSchema=True)
df = df.filter(F.col("tahun") == 2025)

hasil = (
    df
    .withColumn("rata_per_bulan", F.round(F.col("total") / 12))
    .withColumn("kategori",
                F.when(F.col("rata_per_bulan") > 20000, "Tinggi")
                 .when(F.col("rata_per_bulan") >= 10000, "Sedang")
                 .otherwise("Rendah"))
    .orderBy(F.col("total").desc()))

hasil.write.mode("overwrite")\
     .parquet(f"/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/output/wisata_ringkas")

---

# HARI 4 (3 Oktober 2026)

In [0]:
SUMBER = "/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/jumlah-wisatawan-tahun-n-2025.csv"
df = spark.read.csv(SUMBER, header=True, inferSchema=True)
df = df.filter(F.col("tahun") == 2025)

display(df.groupBy("kode_wisata").count().filter(F.col("count") > 1))
df.printSchema()

kode_wisata,count
1.1,2


root
 |-- kode_provinsi: integer (nullable = true)
 |-- nama_provinsi: string (nullable = true)
 |-- kode_kabupaten_kota: integer (nullable = true)
 |-- nama_kabupaten_kota: string (nullable = true)
 |-- kode_wisata: double (nullable = true)
 |-- nama_wisata: string (nullable = true)
 |-- januari: integer (nullable = true)
 |-- februari: integer (nullable = true)
 |-- maret: integer (nullable = true)
 |-- april: integer (nullable = true)
 |-- mei: integer (nullable = true)
 |-- juni: integer (nullable = true)
 |-- juli: integer (nullable = true)
 |-- agustus: integer (nullable = true)
 |-- september: integer (nullable = true)
 |-- oktober: integer (nullable = true)
 |-- november: integer (nullable = true)
 |-- desember: integer (nullable = true)
 |-- total: integer (nullable = true)
 |-- satuan: string (nullable = true)
 |-- tahun: integer (nullable = true)



In [0]:
SUMBER = "/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/jumlah-wisatawan-tahun-n-2025.csv"
df = spark.read.csv(SUMBER, header=True) # tanapa InferSchema
df = (df.withColumn("total", F.col("total").cast("int"))
        .withColumn("tahun", F.col("tahun").cast("int")))

display(df.groupBy("kode_wisata").count().filter(F.col("count") > 1))  # kosong

kode_wisata,count


---

# HARI 6 (5 Oktober 2026)

In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/kuliah_pens/pengembangan_karakter_davin/dataset/"
SUMBER = f"{BASE}/jumlah-wisatawan-tahun-n-2025.csv"
OUTPUT = f"{BASE}/output/wisata_ringkas"

BATAS_TINGGI = 20000   # rata_per_bulan di atas ini = Tinggi
BATAS_SEDANG = 10000   # rata_per_bulan mulai dari ini = Sedang

In [0]:
def extract(path):
    df = spark.read.csv(path, header=True)          # semua kolom dibaca sebagai teks
    return (df.withColumn("total", F.col("total").cast("int"))
              .withColumn("tahun", F.col("tahun").cast("int")))

def transform(df):
    df = df.filter(F.col("tahun") == 2025)          # buang baris(row) yang tahunnya bukan 2025
    return (df
        .withColumn("rata_per_bulan", F.round(F.col("total") / 12))
        .withColumn("kategori",
                    F.when(F.col("rata_per_bulan") > BATAS_TINGGI, "Tinggi")
                     .when(F.col("rata_per_bulan") >= BATAS_SEDANG, "Sedang")
                     .otherwise("Rendah"))
        .select("kode_wisata", "nama_wisata", "total", "rata_per_bulan", "kategori")
        .orderBy(F.col("total").desc()))

def load(df, path):
    df.write.mode("overwrite").parquet(path)

def validate(path):
    hasil = spark.read.parquet(path)                # baca ulang dari output
    assert hasil.count() == 12, "Jumlah baris hasil tidak sesuai."
    assert hasil.select("kode_wisata").distinct().count() == 12, "Kode wisata tidak unik."
    assert hasil.filter(F.col("kategori").isNull()).count() == 0, "Ada kategori kosong."
    print("Validasi lolos.")

In [0]:
raw = extract(SUMBER)
hasil = transform(raw)
load(hasil, OUTPUT)
validate(OUTPUT)
display(hasil)

Validasi lolos.


kode_wisata,nama_wisata,total,rata_per_bulan,kategori
1.1,Wana Wisata Padusan Pacet,351944,29329.0,Tinggi
1.5,Makam Religius Troloyo,259248,21604.0,Tinggi
1.2,Kolam/Pemandian Air Panas,156914,13076.0,Sedang
1.3,Dlundung,85569,7131.0,Rendah
1.4,Jolotundo,74353,6196.0,Rendah
1.6,Museum Trowulan,62236,5186.0,Rendah
1.10,Candi Brahu,26407,2201.0,Rendah
1.9,Candi Bajang Ratu,23842,1987.0,Rendah
1.7,Ubalan,22627,1886.0,Rendah
1.11,Candi Tikus,19239,1603.0,Rendah
